# Stage 7D: verified public OOF blend gate

This standalone Colab notebook aligns the pilkwang package OOF to the ravaghi public base by exact ID and target checks, then evaluates small nested blends. Fleongg is excluded because its package contains inference models but no OOF. No submission is created.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
python = repo_dir / '.venv/bin/python'
subprocess.run(['uv', 'pip', 'install', '--python', str(python), 'kagglehub'], check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
download = subprocess.run([
    str(python), '-c',
    "import kagglehub; print('DATASET_PATH=' + kagglehub.dataset_download('pilkwang/rogii-model-package'))"
], check=True, capture_output=True, text=True)
print(download.stdout)
line = [value for value in download.stdout.splitlines() if value.startswith('DATASET_PATH=')][-1]
package_dir = Path(line.split('=', 1)[1])
assert (package_dir / 'oof/train_gt.parquet').is_file(), package_dir
print('Package:', package_dir)

In [ ]:
BASE_RUN_ID = 'stage7_public_residual_gate_full_v001'
RUN_ID = 'stage7d_public_verified_blend_full_v001'
base_run = artifact_dir / BASE_RUN_ID
run_dir = artifact_dir / RUN_ID
assert (base_run / 'base_oof.parquet').is_file(), base_run
if not (run_dir / 'gate_summary.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-public-blend',
        '--config', 'configs/experiment/public_verified_blend_gate.yaml',
        '--base-run', str(base_run),
        '--package-dir', str(package_dir),
        '--artifact-dir', str(artifact_dir),
        '--run-id', RUN_ID,
    ], cwd=repo_dir, check=True)
else:
    print('Reusing completed run:', run_dir)

In [ ]:
gate = json.loads((run_dir / 'gate_summary.json').read_text())
alignment = json.loads((run_dir / 'alignment.json').read_text())
branches = json.loads((run_dir / 'branch_metrics.json').read_text())
standard = json.loads((run_dir / 'standard_selections.json').read_text())
spatial = json.loads((run_dir / 'spatial_selections.json').read_text())
manifest = json.loads((run_dir / 'blend_manifest.json').read_text())
{
    'promoted': gate['promoted'],
    'alignment': alignment,
    'base_rmse': gate['base_metrics']['pooled_rmse'],
    'nested_candidate_rmse': gate['candidate_metrics']['pooled_rmse'],
    'rmse_delta': gate['pooled_rmse_delta'],
    'bootstrap_95pct': [gate['bootstrap']['ci_2_5'], gate['bootstrap']['ci_97_5']],
    'improved_folds': f"{gate['improved_folds']}/{len(gate['fold_deltas'])}",
    'gates': gate['gates'],
    'spatial_delta': gate['spatial']['pooled_rmse_delta'],
    'branch_metrics': branches,
    'standard_selections': standard,
    'spatial_selections': spatial,
    'inference_spec': manifest['inference_spec'],
}

Proceed to Kaggle integration only when `promoted` is `True`. Exact ID/target alignment is mandatory. The all-OOF best inference spec is reported for packaging but never overrides a failed nested gate.